# DINOv2：三折分组验证 v1

本次不试提交比例，而是建立自己的折外预测。每个病例只由没训练过它所在分组的模型预测。此处“分组”是原 v1 相同报告/完全相同影像的连通分组，不保证患者级隔离。

## 两个 Input 的完整地址

1. 完整 MRI 缓存：https://www.kaggle.com/code/openll/rsna20261010/output
2. 最初 DINOv2 v1 成功输出：https://www.kaggle.com/code/openlm/dinov2-12/output

第二项必须包含 dino-branch-v1/features、dino_encoder.pt、split.csv、config.json。特征来自冻结的通用编码器，可以复用；**不读取 v1 best_head.pt，也不加载两层/四层微调权重**。

## 本次固定实验设计

- 合并原 train/dev/holdout 的 4292 个弱标签病例，按原 group 分三折，种子 20261022；不根据标签选择折分配。
- 58 例官方参考和原 excluded 病例不进入本次训练。所有输入 ID、特征配置与通用权重有一致性检查。
- 每折从新的随机分类头开始，冻结特征训练 20 轮；再从同一通用编码器出发，解冻最后四层微调 6 轮。
- 轮数事先固定，不按外层验证集早停、挑 checkpoint 或挑最优折。每折最终保存冻结头与微调模型的验证预测。
- 输出两套完整 OOF，按同病例比较。过去已查看此数据集并选过模型家族，所以这不是全新的最终独立测试；弱标签也不等于临床真值。
- 本次不能证明与 R6/R7 融合会提分，也不会生成 submission.csv。

## 运行方式

新建 Notebook 上传本文件，**GPU T4 ×2、Internet On、Save Version → Save & Run All**。
Internet 用于安装 timm，模型初始化从 Input 读取。不要覆盖之前的 0.958 提交和训练成果。
这次训练三折，计算量约为单次四层训练的三倍。按此前单次约 20 分钟，粗估约一小时量级，实际以运行日志为准。每 100 批显示进度，每折结束保存权重与小结果包。

## 跑完下载什么

下载 **dino-group-cv-results-small.zip** 到：
`本地实验结果目录/dino-group-cv-results-small.zip`

包内包含 folds.csv、两套 OOF、每折训练/验证 ID、训练日志和 summary.json。确认 summary.status 为 complete 且完成三折；只有 progress.json 不能算全量 OOF。
保留 Output 中每折的 final_model.pt，暂不提交评分。后续任何结构或取片对照应保持这份 folds.csv 不变。


In [ ]:
import subprocess,sys,os,json
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','timm==1.0.29'],check=True)
import torch
assert torch.cuda.is_available(),'请启用 GPU'
print('GPU',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
ROOT=Path('/kaggle/input');dirs=[ROOT];front=[ROOT]
for _ in range(4):
    nxt=[]
    for parent in front:
        for p in parent.iterdir():
            if p.is_dir() and p.name not in ['competitions','rsna-knee-abnormality-detection','train_series','test_series','features','hf-dino-cache']:
                nxt.append(p)
    dirs.extend(nxt);front=nxt
caches=[p for p in dirs if p.name=='rsna-cache-smoke' and (p/'config.json').is_file()]
bases=[p for p in dirs if (p/'features').is_dir() and all((p/f).is_file() for f in ['dino_encoder.pt','split.csv','config.json'])]
assert len(caches)==1,f'需要完整 MRI 缓存，找到 {caches}'
assert len(bases)==1,f'需要最初 v1 完整输出（含 features），找到 {bases}'
CACHE,BASE=caches[0],bases[0]
OUT=Path('/kaggle/working/dino-group-cv-v1')
CODE=Path('/kaggle/working/cv-code');CODE.mkdir(exist_ok=True)
print('CACHE',CACHE,'BASE',BASE,'OUTPUT',OUT,sep='\n')


## 写入训练脚本

In [ ]:
FILES={'dino_branch_train.py': '"""Frozen DINOv2 MRI branch: grouped development and one-time weak holdout."""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\n\nUID = \'StudyInstanceUID\'\nLABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nARCH = \'vit_small_patch14_dinov2.lvd142m\'\n\n\ndef digest(p):\n    return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\n\ndef report_hash(s):\n    s = \' \'.join(unicodedata.normalize(\'NFKC\', str(s)).casefold().split())\n    return hashlib.sha256(s.encode()).hexdigest() if s else \'\'\n\n\ndef make_split(official, weak, image_hashes, seed=20261019):\n    from sklearn.model_selection import GroupShuffleSplit\n    for df in [official, weak]:\n        assert df[UID].notna().all() and df[UID].is_unique, \'Invalid study IDs\'\n    table = official[[UID,\'Report\']+LABELS].copy().reset_index(drop=True)\n    gold = table[LABELS].notna().all(1)\n    empty = table[LABELS].isna().all(1)\n    assert (gold | empty).all(), \'Unexpected partially official-labelled rows\'\n    parent = list(range(len(table)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    seen = {}\n    reports = table.Report.fillna(\'\').map(report_hash)\n    for i, uid in enumerate(table[UID]):\n        for key in [(\'report\',reports[i]),(\'image\',image_hashes[uid])]:\n            if not key[1]: continue\n            if key in seen: parent[find(i)] = find(seen[key])\n            else: seen[key] = i\n    table[\'group\'] = [find(i) for i in range(len(table))]\n    gold_groups = set(table.loc[gold,\'group\'])\n    table[\'split\'] = \'excluded\'\n    table.loc[gold,\'split\'] = \'gold_reference\'\n    eligible = empty & ~table.group.isin(gold_groups) & reports.ne(\'\')\n    aligned = weak.set_index(UID).reindex(table[UID])\n    for c in LABELS:\n        v = aligned[c+\'__verdict\'].astype(\'string\').str.strip().str.upper()\n        assert (v.isna() | v.isin([\'YES\',\'NO\',\'UNK\'])).all(), c\n        vals = v.map({\'YES\':1.,\'NO\':0.,\'UNK\':np.nan}).to_numpy(dtype=float,na_value=np.nan)\n        table.loc[empty,c] = vals[empty.to_numpy()]\n    eligible &= table[LABELS].notna().any(axis=1)\n    ids = np.flatnonzero(eligible)\n    a,b = next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=seed).split(ids,groups=table.loc[ids,\'group\']))\n    pool,hold = ids[a],ids[b]\n    a,b = next(GroupShuffleSplit(n_splits=1,test_size=.15/.85,random_state=seed+1).split(pool,groups=table.loc[pool,\'group\']))\n    for name, rows in [(\'train\',pool[a]),(\'dev\',pool[b]),(\'holdout\',hold)]: table.loc[rows,\'split\'] = name\n    sets = [set(table.loc[table.split.eq(s),\'group\']) for s in [\'train\',\'dev\',\'holdout\',\'gold_reference\']]\n    assert all(not sets[i]&sets[j] for i in range(4) for j in range(i))\n    return table.drop(columns=\'Report\')\n\n\nclass Images(Dataset):\n    def __init__(self, ids, cache): self.ids,self.cache = list(ids),Path(cache)\n    def __len__(self): return len(self.ids)\n    def __getitem__(self,i):\n        uid = self.ids[i]\n        with np.load(self.cache/(uid+\'.npz\'),allow_pickle=False) as z:\n            x,m = z[\'x\'].copy(),z[\'mask\'].astype(bool)\n        assert x.shape==(3,12,224,224) and x.dtype==np.uint8, (uid,x.shape,x.dtype)\n        assert m.shape==(3,) and m.any(), uid\n        h=hashlib.sha256(x.tobytes()+m.tobytes()).hexdigest()\n        return torch.from_numpy(x),torch.from_numpy(m),uid,h\n\n\ndef extract(ids,cache,out,device):\n    import timm\n    fdir=out/\'features\'; fdir.mkdir(exist_ok=True)\n    model=timm.create_model(ARCH,pretrained=True,num_classes=0,img_size=224).eval().to(device)\n    cfg=timm.data.resolve_model_data_config(model)\n    torch.save(model.cpu().state_dict(),out/\'dino_encoder.pt\'); model.to(device)\n    config={\'arch\':ARCH,\'size\':224,\'slices\':12,\'planes\':3,\'mean\':list(cfg[\'mean\']),\'std\':list(cfg[\'std\']),\n            \'encoder_sha256\':digest(out/\'dino_encoder.pt\'),\'cache_config_sha256\':digest(cache/\'config.json\'),\n            \'timm\':timm.__version__,\'torch\':torch.__version__,\'input\':\'uint8 / 255; grayscale repeated RGB; no spatial crop\'}\n    # Never silently reuse features from a different encoder or preprocessing recipe.\n    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()\n    config[\'signature\']=signature\n    (out/\'feature_config.json\').write_text(json.dumps(config,indent=2))\n    hashes={}; todo=[]\n    for uid in ids:\n        p=fdir/(uid+\'.npz\')\n        if p.exists():\n            with np.load(p,allow_pickle=False) as z:\n                if str(z[\'signature\'])==signature:\n                    hashes[uid]=str(z[\'image_hash\']); continue\n        todo.append(uid)\n    mean=torch.tensor(cfg[\'mean\'],device=device)[None,:,None,None]\n    std=torch.tensor(cfg[\'std\'],device=device)[None,:,None,None]\n    if torch.cuda.device_count()>1: model=nn.DataParallel(model)\n    loader=DataLoader(Images(todo,cache),batch_size=4,num_workers=2,pin_memory=True)\n    start=time.time(); done=0\n    with torch.inference_mode():\n        for x,m,uids,hh in loader:\n            b=len(uids)\n            images=x.to(device).reshape(-1,1,224,224).float().div_(255).expand(-1,3,-1,-1)\n            images=(images-mean)/std\n            with torch.autocast(\'cuda\',dtype=torch.float16): f=model(images)\n            f=f.float().reshape(b,3,12,-1).cpu().numpy().astype(\'float16\')\n            assert np.isfinite(f).all()\n            for j,uid in enumerate(uids):\n                p=fdir/(uid+\'.npz\'); tmp=p.with_suffix(\'.tmp.npz\')\n                np.savez_compressed(tmp,f=f[j],mask=m[j].numpy(),image_hash=hh[j],signature=signature)\n                tmp.replace(p);hashes[uid]=hh[j]\n            done+=b\n            if done%40==0 or done==len(todo):\n                elapsed=time.time()-start\n                print(f\'FEATURES {done}/{len(todo)} elapsed={elapsed/60:.1f} min ETA={(len(todo)-done)*elapsed/max(done,1)/60:.1f} min\',flush=True)\n    del model;torch.cuda.empty_cache()\n    return hashes,config\n\n\nclass Features(Dataset):\n    def __init__(self,df,folder):\n        self.df=df.reset_index(drop=True); self.folder=folder\n    def __len__(self): return len(self.df)\n    def __getitem__(self,i):\n        row=self.df.iloc[i]\n        with np.load(self.folder/(row[UID]+\'.npz\'),allow_pickle=False) as z:\n            f=torch.from_numpy(z[\'f\'].astype(\'float32\'));m=torch.from_numpy(z[\'mask\'].copy())\n        y=torch.tensor(row[LABELS].to_numpy(dtype=\'float32\'))\n        return f,m,y\n\n\nclass Head(nn.Module):\n    def __init__(self,dim=384):\n        super().__init__()\n        self.proj=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,128),nn.GELU())\n        self.position=nn.Parameter(torch.zeros(1,3,12,128))\n        self.attn=nn.Linear(128,12)\n        self.value=nn.Linear(128,12)\n        self.bias=nn.Parameter(torch.zeros(12))\n        self.drop=nn.Dropout(.2)\n    def forward(self,f,mask):\n        h=self.proj(f)+self.position\n        b=h.shape[0]; h=self.drop(h).reshape(b,36,128)\n        good=mask[:,:,None].expand(-1,-1,12).reshape(b,36,1)\n        a=self.attn(h).masked_fill(~good,-1e4).softmax(1)\n        return (a*self.value(h)).sum(1)+self.bias\n\n\ndef loss_fn(z,y):\n    valid=torch.isfinite(y)\n    loss=nn.functional.binary_cross_entropy_with_logits(z,torch.nan_to_num(y),reduction=\'none\')\n    counts=valid.sum(0); usable=counts>0\n    return ((loss*valid).sum(0)/counts.clamp_min(1))[usable].mean()\n\n\ndef metrics(y,p):\n    from sklearn.metrics import roc_auc_score\n    result={}\n    for j,c in enumerate(LABELS):\n        m=np.isfinite(y[:,j]); truth=y[m,j]\n        result[c]=float(roc_auc_score(truth,p[m,j])) if len(np.unique(truth))==2 else None\n    available=[v for v in result.values() if v is not None]\n    return {\'macro_auc\':float(np.mean(available)) if available else None,\'valid_classes\':len(available),\'per_class\':result}\n\n\n@torch.inference_mode()\ndef evaluate(model,loader,device):\n    model.eval(); yy=[]; pp=[]\n    for f,m,y in loader:\n        pp.append(model(f.to(device),m.to(device)).sigmoid().cpu().numpy());yy.append(y.numpy())\n    y,p=np.concatenate(yy),np.concatenate(pp)\n    return metrics(y,p),p\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    for name in [\'data\',\'weak\',\'cache\',\'out\']: ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--epochs\',type=int,default=20);a=ap.parse_args()\n    assert torch.cuda.is_available(),\'Enable GPU\'\n    seed=20261019;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(4)\n    a.out.mkdir(parents=True,exist_ok=True)\n    if (a.out/\'summary.json\').exists(): raise RuntimeError(\'Completed experiment exists; use a fresh output directory\')\n    started=time.time()\n    official=pd.read_csv(a.data/\'train.csv\',dtype={UID:str});weak=pd.read_csv(a.weak,dtype={UID:str})\n    assert official[UID].is_unique and official[UID].notna().all()\n    for c in LABELS:\n        assert official[c].dropna().isin([0,1]).all()\n        assert c+\'__verdict\' in weak\n    assert all((a.cache/(u+\'.npz\')).is_file() for u in official[UID]),\'Incomplete cache; mount full 4407-study output\'\n    hashes,fcfg=extract(official[UID],a.cache,a.out,\'cuda\')\n    split=make_split(official,weak,hashes,seed)\n    split.to_csv(a.out/\'split.csv\',index=False)\n    coverage={s:{c:{\'positive\':int((d[c]==1).sum()),\'negative\':int((d[c]==0).sum()),\'unknown\':int(d[c].isna().sum())} for c in LABELS} for s,d in split.groupby(\'split\')}\n    (a.out/\'label_coverage.json\').write_text(json.dumps(coverage,indent=2))\n    counts=split.split.value_counts().to_dict();print(\'SPLITS\',counts,flush=True)\n    for c,v in coverage[\'dev\'].items():\n        if not v[\'positive\'] or not v[\'negative\']: raise RuntimeError(f\'Dev lacks both classes: {c}; do not select a different seed based on results\')\n    config={\'seed\':seed,\'epochs_max\':a.epochs,\'patience\':5,\'lr\':.001,\'weight_decay\':.01,\'backbone_frozen\':True,\n            \'selection\':\'dev weak macro AUC only\',\'feature_config\':fcfg,\'official_sha256\':digest(a.data/\'train.csv\'),\n            \'weak_sha256\':digest(a.weak),\'script_sha256\':digest(__file__),\'split_sha256\':digest(a.out/\'split.csv\'),\n            \'grouping\':\'connected components of normalized exact report and exact image hashes; not patient-level IDs\',\n            \'weak_holdout_is_clinical_ground_truth\':False,\'gold_independent_test\':False,\'baseline_complementarity_tested\':False}\n    (a.out/\'config.json\').write_text(json.dumps(config,indent=2))\n    loaders={s:DataLoader(Features(split[split.split.eq(s)],a.out/\'features\'),batch_size=64,shuffle=s==\'train\',num_workers=2) for s in [\'train\',\'dev\',\'holdout\',\'gold_reference\']}\n    model=Head().cuda();opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)\n    best=-float(\'inf\'); stale=0\n    for epoch in range(1,a.epochs+1):\n        model.train();total=0.;steps=0\n        for f,m,y in loaders[\'train\']:\n            opt.zero_grad(set_to_none=True)\n            loss=loss_fn(model(f.cuda(),m.cuda()),y.cuda())\n            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()\n            total+=loss.item();steps+=1\n        score,_=evaluate(model,loaders[\'dev\'],\'cuda\')\n        record={\'epoch\':epoch,\'loss\':total/steps,\'dev\':score}\n        with (a.out/\'metrics.jsonl\').open(\'a\') as h:h.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record),flush=True)\n        if score[\'macro_auc\']>best:\n            best=score[\'macro_auc\'];stale=0\n            torch.save({\'state_dict\':model.state_dict(),\'epoch\':epoch,\'labels\':LABELS,\'feature_signature\':fcfg[\'signature\']},a.out/\'best_head.pt\')\n        else:stale+=1\n        if stale>=5:break\n    ckpt=torch.load(a.out/\'best_head.pt\',map_location=\'cuda\',weights_only=True);model.load_state_dict(ckpt[\'state_dict\'])\n    scores={}\n    # The holdout and gold are first evaluated only after the dev-selected model is frozen.\n    for name in [\'dev\',\'holdout\',\'gold_reference\']:\n        scores[name],pred=evaluate(model,loaders[name],\'cuda\')\n        rows=split.loc[split.split.eq(name),[UID,\'group\']+LABELS].reset_index(drop=True)\n        rows=rows.rename(columns={c:\'target::\'+c for c in LABELS})\n        for j,c in enumerate(LABELS):rows[\'pred::\'+c]=pred[:,j]\n        rows.to_csv(a.out/(name+\'_predictions.csv\'),index=False)\n    result={\'status\':\'training_complete\',\'selected_epoch\':ckpt[\'epoch\'],\'counts\':counts,\'metrics\':scores,\n            \'elapsed_seconds\':time.time()-started,\'submission_ready\':False,\'complementarity_with_r6_r7_unknown\':True}\n    (a.out/\'summary.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2),flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'dino_finetune_four_blocks.py': '"""Controlled last-four-block fine-tuning from the frozen-feature experiment."""\nimport argparse\nimport json\nimport math\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom dino_branch_train import ARCH, UID, LABELS, Head, Images, digest, loss_fn, metrics\n\n\ndef unfreeze_tail(encoder):\n    encoder.requires_grad_(False)\n    assert len(encoder.blocks) >= 4\n    for block in encoder.blocks[-4:]: block.requires_grad_(True)\n    encoder.norm.requires_grad_(True)\n    return [n for n,p in encoder.named_parameters() if p.requires_grad]\n\n\nclass Model(nn.Module):\n    def __init__(self,encoder,head,mean,std):\n        super().__init__(); self.encoder=encoder;self.head=head\n        self.register_buffer(\'mean\',torch.tensor(mean)[None,:,None,None])\n        self.register_buffer(\'std\',torch.tensor(std)[None,:,None,None])\n    def train(self,mode=True):\n        super().train(mode)\n        # Keep encoder stochastic layers disabled; gradients still flow through its tail.\n        self.encoder.eval()\n        return self\n    def forward(self,x,mask):\n        b=x.shape[0]\n        x=x.reshape(-1,1,224,224).float().div(255).expand(-1,3,-1,-1)\n        x=(x-self.mean)/self.std\n        with torch.autocast(device_type=x.device.type,dtype=torch.float16,enabled=x.is_cuda):\n            f=self.encoder(x)\n        # Same FP16 feature rounding and FP32 head as v1; cast remains differentiable.\n        f=f.half().float().reshape(b,3,12,-1)\n        return self.head(f,mask)\n\n\nclass Studies(Images):\n    def __init__(self,df,cache):\n        self.df=df.reset_index(drop=True)\n        super().__init__(self.df[UID],cache)\n    def __getitem__(self,i):\n        x,m,uid,h=super().__getitem__(i)\n        return x,m,torch.tensor(self.df.loc[i,LABELS].to_numpy(dtype=\'float32\'))\n\n\n@torch.inference_mode()\ndef evaluate(model,loader,device):\n    model.eval();ys=[];ps=[];t=time.time()\n    for i,(x,m,y) in enumerate(loader):\n        ps.append(model(x.to(device),m.to(device)).sigmoid().cpu().numpy());ys.append(y.numpy())\n        if (i+1)%100==0: print(f\'EVAL batches {i+1}/{len(loader)} elapsed {(time.time()-t)/60:.1f} min\',flush=True)\n    y,p=np.concatenate(ys),np.concatenate(ps)\n    return metrics(y,p),p\n\n\ndef save_predictions(rows,p,path):\n    f=rows[[UID,\'group\']+LABELS].reset_index(drop=True).rename(columns={c:\'target::\'+c for c in LABELS})\n    for j,c in enumerate(LABELS):f[\'pred::\'+c]=p[:,j]\n    f.to_csv(path,index=False)\n\n\ndef main():\n    import timm\n    ap=argparse.ArgumentParser()\n    for name in [\'base\',\'cache\',\'out\']:ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--epochs\',type=int,default=6);args=ap.parse_args()\n    assert torch.cuda.is_available(),\'Enable GPU\'\n    if args.out.exists() and any(args.out.iterdir()):raise RuntimeError(\'Use an empty output directory; do not mix runs\')\n    args.out.mkdir(parents=True,exist_ok=True);out=args.out;start=time.time()\n    seed=20261020;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);torch.set_num_threads(4)\n    cfg=json.loads((args.base/\'config.json\').read_text());fcfg=cfg[\'feature_config\']\n    assert fcfg[\'arch\']==ARCH and fcfg[\'size\']==224\n    assert timm.__version__==fcfg[\'timm\'],\'Use same timm version as v1\'\n    for f,key in [(\'split.csv\',\'split_sha256\')]:assert digest(args.base/f)==cfg[key]\n    assert digest(args.base/\'dino_encoder.pt\')==fcfg[\'encoder_sha256\']\n    assert digest(args.cache/\'config.json\')==fcfg[\'cache_config_sha256\']\n    split=pd.read_csv(args.base/\'split.csv\',dtype={UID:str})\n    assert split[UID].is_unique\n    active=split[split.split.ne(\'excluded\')]\n    assert active.groupby(\'group\').split.nunique().max()==1\n    for uid in active[UID]:assert (args.cache/(uid+\'.npz\')).is_file(),uid\n    shutil.copyfile(args.base/\'split.csv\',out/\'split.csv\')\n    encoder=timm.create_model(ARCH,pretrained=False,num_classes=0,img_size=224)\n    encoder.load_state_dict(torch.load(args.base/\'dino_encoder.pt\',weights_only=True,map_location=\'cpu\'),strict=True)\n    head=Head();old=torch.load(args.base/\'best_head.pt\',weights_only=True,map_location=\'cpu\')\n    assert old[\'labels\']==LABELS and old[\'feature_signature\']==fcfg[\'signature\']\n    head.load_state_dict(old[\'state_dict\'],strict=True)\n    names=unfreeze_tail(encoder)\n    raw=Model(encoder,head,fcfg[\'mean\'],fcfg[\'std\']).cuda()\n    model=nn.DataParallel(raw) if torch.cuda.device_count()>1 else raw\n    dfs={s:split[split.split.eq(s)].copy() for s in [\'train\',\'dev\',\'holdout\',\'gold_reference\']}\n    loaders={s:DataLoader(Studies(df,args.cache),batch_size=2,shuffle=s==\'train\',num_workers=2,pin_memory=True) for s,df in dfs.items()}\n    audit={\'seed\':seed,\'epochs_max\':args.epochs,\'encoder_lr\':1e-5,\'head_lr\':1e-4,\'weight_decay\':.01,\n           \'unfrozen_blocks\':4,\'unfrozen_encoder_parameters\':names,\'batch_studies\':2,\'gradient_accumulation\':4,\n           \'input_recipe\':\'same 36 slices as v1; no new augmentation or crop\',\'selection\':\'dev only; epoch zero eligible\',\n           \'holdout_previously_inspected\':True,\'gold_independent_test\':False,\'baseline_complementarity_tested\':False,\n           \'base_config\':cfg,\'base_head_sha256\':digest(args.base/\'best_head.pt\'),\'script_sha256\':digest(__file__),\n           \'helper_sha256\':digest(Path(__file__).with_name(\'dino_branch_train.py\'))}\n    (out/\'config.json\').write_text(json.dumps(audit,indent=2))\n    print(\'BASELINE reconstruction on dev\',flush=True)\n    baseline,p=evaluate(model,loaders[\'dev\'],\'cuda\')\n    previous=pd.read_csv(args.base/\'dev_predictions.csv\',dtype={UID:str}).set_index(UID)\n    expected=previous.loc[dfs[\'dev\'][UID],[\'pred::\'+c for c in LABELS]].to_numpy()\n    delta=float(np.max(np.abs(p-expected)))\n    original=json.loads((args.base/\'summary.json\').read_text())[\'metrics\'][\'dev\'][\'macro_auc\']\n    check={\'dev\':baseline,\'max_probability_difference\':delta,\'original_dev_auc\':original}\n    (out/\'baseline_check.json\').write_text(json.dumps(check,indent=2))\n    assert delta<.003 and abs(baseline[\'macro_auc\']-original)<.002,\'Baseline mismatch: inspect baseline_check.json; training stopped\'\n    save_predictions(dfs[\'dev\'],p,out/\'baseline_dev_predictions.csv\')\n    best=baseline[\'macro_auc\'];best_epoch=0;stale=0;history=[]\n    def save(epoch):\n        # Detached CPU copies avoid a large GPU checkpoint allocation.\n        torch.save({\'encoder\':{k:v.detach().cpu() for k,v in raw.encoder.state_dict().items()},\n                    \'head\':{k:v.detach().cpu() for k,v in raw.head.state_dict().items()},\n                    \'epoch\':epoch,\'labels\':LABELS,\'feature_config\':fcfg},out/\'best_model.pt\')\n    save(0)\n    opt=torch.optim.AdamW([{\'params\':[p for p in raw.encoder.parameters() if p.requires_grad],\'lr\':1e-5},\n                          {\'params\':raw.head.parameters(),\'lr\':1e-4}],weight_decay=.01)\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=args.epochs,eta_min=0.)\n    scaler=torch.amp.GradScaler(\'cuda\');limit=7.5*3600\n    for epoch in range(1,args.epochs+1):\n        # Leave room for final evaluation and packaging; never start an obviously over-budget epoch.\n        if history and time.time()-start+history[-1][\'seconds\']+900>limit:break\n        t=time.time();model.train();opt.zero_grad(set_to_none=True);losses=[];aborted=False\n        for i,(x,m,y) in enumerate(loaders[\'train\']):\n            if time.time()-start>limit:aborted=True;break\n            n=len(loaders[\'train\']);group_size=min(4,n-(i//4)*4)\n            loss=loss_fn(model(x.cuda(),m.cuda()),y.cuda())\n            assert torch.isfinite(loss),\'Nonfinite loss\'\n            scaler.scale(loss/group_size).backward();losses.append(float(loss.detach()))\n            if (i+1)%4==0 or i+1==n:\n                scaler.unscale_(opt);nn.utils.clip_grad_norm_([p for p in raw.parameters() if p.requires_grad],1.)\n                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)\n            if (i+1)%100==0:\n                elapsed=time.time()-t\n                print(f\'EPOCH {epoch} batches {i+1}/{n} loss {np.mean(losses):.4f} ETA {(n-i-1)*elapsed/(i+1)/60:.1f} min\',flush=True)\n        if aborted:\n            print(\'TIME BUDGET: incomplete epoch discarded; restoring best checkpoint\',flush=True);break\n        score,p=evaluate(model,loaders[\'dev\'],\'cuda\');scheduler.step()\n        rec={\'epoch\':epoch,\'loss\':float(np.mean(losses)),\'dev\':score,\'seconds\':time.time()-t}\n        history.append(rec)\n        with (out/\'metrics.jsonl\').open(\'a\') as f:f.write(json.dumps(rec)+\'\\n\')\n        print(json.dumps(rec),flush=True)\n        if score[\'macro_auc\']>best:\n            best=score[\'macro_auc\'];best_epoch=epoch;stale=0;save(epoch)\n        else:stale+=1\n        if stale>=3:break\n    ck=torch.load(out/\'best_model.pt\',weights_only=True,map_location=\'cpu\')\n    raw.encoder.load_state_dict(ck[\'encoder\']);raw.head.load_state_dict(ck[\'head\']);del ck\n    scores={}\n    # No holdout or gold measurements were used for model selection.\n    for s in [\'dev\',\'holdout\',\'gold_reference\']:\n        scores[s],p=evaluate(model,loaders[s],\'cuda\');save_predictions(dfs[s],p,out/(s+\'_predictions.csv\'))\n    result={\'status\':\'complete\',\'selected_epoch\':best_epoch,\'epochs_completed\':len(history),\'baseline_dev_auc\':baseline[\'macro_auc\'],\n            \'dev_gain\':best-baseline[\'macro_auc\'],\'metrics\':scores,\'counts\':{s:len(d) for s,d in dfs.items()},\n            \'elapsed_seconds\':time.time()-start,\'submission_ready\':False,\'holdout_previously_inspected\':True}\n    (out/\'summary.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2),flush=True)\n    import zipfile\n    with zipfile.ZipFile(out.parent/\'dino-four-block-results-small.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in out.iterdir():\n            if p.suffix in [\'.json\',\'.jsonl\',\'.csv\']:z.write(p,p.name)\n    print(\'RESULTS READY: dino-four-block-results-small.zip\',flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'dino_group_cv.py': '"""Three-fold grouped weak-label CV. Fixed schedules, no outer-fold selection."""\nimport argparse\nimport gc\nimport json\nimport random\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom dino_branch_train import UID,LABELS,ARCH,Head,Features,loss_fn,metrics,digest,evaluate as evaluate_features\nfrom dino_finetune_four_blocks import Model,Studies,unfreeze_tail,evaluate,save_predictions\n\n\ndef assign_folds(frame,seed=20261022,nfold=3):\n    out=frame[frame.split.isin([\'train\',\'dev\',\'holdout\'])].copy().reset_index(drop=True)\n    assert out[UID].notna().all() and out[UID].is_unique\n    assert out.group.notna().all() and out[LABELS].notna().any(axis=1).all()\n    groups=out.groupby(\'group\',sort=True).size()\n    assert len(groups)>=nfold\n    rng=np.random.default_rng(seed)\n    # Random tie order, then greedily balance group sizes without consulting labels.\n    order=list(groups.index[rng.permutation(len(groups))]);order.sort(key=lambda g:-groups[g])\n    totals=[0]*nfold;mapping={}\n    for g in order:\n        f=int(np.argmin(totals));mapping[g]=f;totals[f]+=int(groups[g])\n    out[\'fold\']=out.group.map(mapping).astype(int)\n    assert out.groupby(\'group\').fold.nunique().max()==1\n    return out\n\n\ndef package(out):\n    with zipfile.ZipFile(out.parent/\'dino-group-cv-results-small.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in out.rglob(\'*\'):\n            if p.is_file() and p.suffix in [\'.csv\',\'.json\',\'.jsonl\']:z.write(p,p.relative_to(out))\n\n\ndef main():\n    import timm\n    ap=argparse.ArgumentParser()\n    for k in [\'base\',\'cache\',\'out\']:ap.add_argument(\'--\'+k,type=Path,required=True)\n    a=ap.parse_args();assert torch.cuda.is_available()\n    assert not a.out.exists() or not any(a.out.iterdir()),\'Use a new empty output directory\'\n    a.out.mkdir(parents=True,exist_ok=True);start=time.time();torch.set_num_threads(4)\n    source=json.loads((a.base/\'config.json\').read_text());fc=source[\'feature_config\']\n    assert source[\'backbone_frozen\'] is True,\'Must use v1 generic encoder, not fine-tuned checkpoint\'\n    assert timm.__version__==fc[\'timm\'] and fc[\'arch\']==ARCH\n    assert digest(a.base/\'dino_encoder.pt\')==fc[\'encoder_sha256\']\n    assert digest(a.base/\'split.csv\')==source[\'split_sha256\']\n    assert digest(a.cache/\'config.json\')==fc[\'cache_config_sha256\']\n    frame=pd.read_csv(a.base/\'split.csv\',dtype={UID:str});folds=assign_folds(frame)\n    assert not set(folds.group)&set(frame.loc[frame.split.eq(\'gold_reference\'),\'group\'])\n    folds.to_csv(a.out/\'folds.csv\',index=False)\n    for uid in folds[UID]:\n        assert (a.cache/(uid+\'.npz\')).is_file(),uid\n        with np.load(a.base/\'features\'/(uid+\'.npz\'),allow_pickle=False) as z:\n            assert str(z[\'signature\'])==fc[\'signature\'] and z[\'f\'].shape==(3,12,384)\n            assert np.isfinite(z[\'f\']).all()\n    config={\'seed\':20261022,\'nfold\':3,\'head_epochs\':20,\'finetune_epochs\':6,\'unfrozen_blocks\':4,\n            \'head_warmup_lr\':.001,\'finetune_head_lr\':1e-4,\'encoder_lr\':1e-5,\n            \'selection\':\'fixed final epoch, no outer fold early stopping or checkpoint selection\',\n            \'head_initialization\':\'new random head per fold; never load v1 best_head.pt\',\n            \'encoder_initialization\':\'same generic pretrained v1 encoder per fold, never a knee-finetuned checkpoint\',\n            \'excluded\':\'gold_reference and excluded rows from v1\',\n            \'limitations\':[\'weak labels, not clinical ground truth\',\'grouping is exact report/image, not patient identity\',\n                          \'dataset and model family already used for development; not a pristine final test\',\n                          \'cannot establish complementarity with R6/R7 without their matched OOF\'],\n            \'feature_config\':fc,\'folds_sha256\':digest(a.out/\'folds.csv\'),\'script_sha256\':digest(__file__)}\n    (a.out/\'config.json\').write_text(json.dumps(config,indent=2))\n    print(\'FOLD COUNTS\',folds.fold.value_counts().sort_index().to_dict(),flush=True)\n    results=[];all_frozen=[];all_fine=[]\n    for fold in range(3):\n        t=time.time();seed=20261022+fold\n        random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n        train=folds[folds.fold.ne(fold)].copy();val=folds[folds.fold.eq(fold)].copy()\n        assert set(train.group).isdisjoint(val.group) and set(train[UID]).isdisjoint(val[UID])\n        folder=a.out/f\'fold{fold}\';folder.mkdir()\n        train[[UID,\'group\',\'fold\']].to_csv(folder/\'training_ids.csv\',index=False)\n        val[[UID,\'group\',\'fold\']].to_csv(folder/\'validation_ids.csv\',index=False)\n        head=Head().cuda();opt=torch.optim.AdamW(head.parameters(),lr=.001,weight_decay=.01)\n        fl=DataLoader(Features(train,a.base/\'features\'),batch_size=64,shuffle=True,num_workers=2)\n        for epoch in range(20):\n            head.train();losses=[]\n            for f,m,y in fl:\n                opt.zero_grad(set_to_none=True);loss=loss_fn(head(f.cuda(),m.cuda()),y.cuda())\n                loss.backward();nn.utils.clip_grad_norm_(head.parameters(),1.);opt.step();losses.append(loss.item())\n            print(f\'FOLD {fold} HEAD {epoch+1}/20 train_loss={np.mean(losses):.5f}\',flush=True)\n        vl=DataLoader(Features(val,a.base/\'features\'),batch_size=64,num_workers=2)\n        frozen_score,p=evaluate_features(head,vl,\'cuda\')\n        save_predictions(val,p,folder/\'frozen_predictions.csv\')\n        torch.save({\'state_dict\':head.state_dict(),\'labels\':LABELS,\'fold\':fold},folder/\'frozen_head.pt\')\n        del opt,fl,vl\n        encoder=timm.create_model(ARCH,pretrained=False,num_classes=0,img_size=224)\n        encoder.load_state_dict(torch.load(a.base/\'dino_encoder.pt\',map_location=\'cpu\',weights_only=True),strict=True)\n        unfreeze_tail(encoder)\n        raw=Model(encoder,head,fc[\'mean\'],fc[\'std\']).cuda()\n        model=nn.DataParallel(raw) if torch.cuda.device_count()>1 else raw\n        loader=DataLoader(Studies(train,a.cache),batch_size=2,shuffle=True,num_workers=2,pin_memory=True)\n        opt=torch.optim.AdamW([{\'params\':[p for p in raw.encoder.parameters() if p.requires_grad],\'lr\':1e-5},\n                              {\'params\':raw.head.parameters(),\'lr\':1e-4}],weight_decay=.01)\n        sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=6)\n        scaler=torch.amp.GradScaler(\'cuda\')\n        for epoch in range(6):\n            model.train();opt.zero_grad(set_to_none=True);losses=[];ep_start=time.time()\n            for i,(x,m,y) in enumerate(loader):\n                loss=loss_fn(model(x.cuda(),m.cuda()),y.cuda());assert torch.isfinite(loss)\n                group_size=min(4,len(loader)-(i//4)*4)\n                scaler.scale(loss/group_size).backward();losses.append(float(loss.detach()))\n                if (i+1)%4==0 or i+1==len(loader):\n                    scaler.unscale_(opt);nn.utils.clip_grad_norm_([p for p in raw.parameters() if p.requires_grad],1.)\n                    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)\n                if (i+1)%100==0:\n                    print(f\'FOLD {fold} FINETUNE {epoch+1}/6 batch {i+1}/{len(loader)} elapsed={(time.time()-ep_start)/60:.1f} min\',flush=True)\n            sched.step()\n            rec={\'fold\':fold,\'epoch\':epoch+1,\'train_loss\':float(np.mean(losses)),\'seconds\':time.time()-ep_start}\n            with (folder/\'train_metrics.jsonl\').open(\'a\') as f:f.write(json.dumps(rec)+\'\\n\')\n        # Only now evaluate the outer fold: no outer labels used for optimization or epoch selection.\n        valid_loader=DataLoader(Studies(val,a.cache),batch_size=2,num_workers=2,pin_memory=True)\n        fine_score,p=evaluate(model,valid_loader,\'cuda\');save_predictions(val,p,folder/\'finetuned_predictions.csv\')\n        torch.save({\'encoder\':{k:v.detach().cpu() for k,v in raw.encoder.state_dict().items()},\n                    \'head\':{k:v.detach().cpu() for k,v in raw.head.state_dict().items()},\n                    \'fold\':fold,\'epoch\':6,\'labels\':LABELS,\'feature_config\':fc},folder/\'final_model.pt\')\n        result={\'fold\':fold,\'train\':len(train),\'validation\':len(val),\'frozen\':frozen_score,\'finetuned\':fine_score,\'seconds\':time.time()-t}\n        (folder/\'summary.json\').write_text(json.dumps(result,indent=2));results.append(result)\n        print(\'FOLD COMPLETE\',json.dumps(result),flush=True)\n        for name,collector in [(\'frozen\',all_frozen),(\'finetuned\',all_fine)]:\n            table=pd.read_csv(folder/(name+\'_predictions.csv\'),dtype={UID:str});table[\'fold\']=fold;collector.append(table)\n        (a.out/\'progress.json\').write_text(json.dumps({\'completed_folds\':len(results),\'required_folds\':3},indent=2));package(a.out)\n        del model,raw,head,encoder,opt,sched,scaler,loader,valid_loader\n        gc.collect();torch.cuda.empty_cache()\n    summary={\'status\':\'complete\',\'folds\':results,\'submission_ready\':False,\'elapsed_seconds\':time.time()-start,\'pooled_oof\':{}}\n    for name,tables in [(\'frozen\',all_frozen),(\'finetuned\',all_fine)]:\n        df=pd.concat(tables,ignore_index=True);assert df[UID].is_unique and set(df[UID])==set(folds[UID])\n        df.to_csv(a.out/(name+\'_oof.csv\'),index=False)\n        summary[\'pooled_oof\'][name]=metrics(df[[\'target::\'+c for c in LABELS]].to_numpy(),df[[\'pred::\'+c for c in LABELS]].to_numpy())\n    summary[\'mean_fold_macro_auc\']={name:float(np.mean([r[name][\'macro_auc\'] for r in results])) for name in [\'frozen\',\'finetuned\']}\n    (a.out/\'summary.json\').write_text(json.dumps(summary,indent=2));package(a.out)\n    print(json.dumps(summary,indent=2),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
for name,text in FILES.items():(CODE/name).write_text(text,encoding='utf8')


## 运行三折
外层验证结果不会用于训练或挑轮次，不需要手动选择任何最佳轮。

In [ ]:
cmd=[sys.executable,'-u',str(CODE/'dino_group_cv.py'),'--base',str(BASE),'--cache',str(CACHE),'--out',str(OUT)]
env=dict(os.environ,PYTHONPATH=str(CODE),PYTHONUNBUFFERED='1',HF_HUB_OFFLINE='1')
with Path('/kaggle/working/dino-group-cv.log').open('w') as log:
    with subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        rc=proc.wait()
assert rc==0,'运行失败：保留日志和已完成折，不要误认为全量完成'
summary=json.loads((OUT/'summary.json').read_text())
assert summary['status']=='complete' and len(summary['folds'])==3
print('三折完成。请下载小结果包，暂不提交。')


## 下载小结果包

In [ ]:
from IPython.display import FileLink,display
os.chdir('/kaggle/working')
assert Path('dino-group-cv-results-small.zip').is_file()
display(FileLink('dino-group-cv-results-small.zip'))
